# 01. Khám phá và Trực quan hóa Dữ liệu (Exploratory Data Analysis - EDA)
Notebook này thực hiện khảo sát chi tiết bộ dữ liệu `clickbait_dataset_vietnamese.csv` gồm **3,414 bài báo tiếng Việt** từ 8 nguồn báo lớn:
1. **Phân phối nhãn (`label`)**: Kiểm tra mức độ mất cân bằng dữ liệu giữa `clickbait` và `non-clickbait`.
2. **Phân phối theo Nguồn báo (`source`) và Chuyên mục (`category`)**: Phân tích hiện tượng thiên lệch nguồn (Source Bias) và lý giải tại sao cần loại bỏ `source` khỏi tập đặc trưng huấn luyện nhằm tránh *Data Leakage / Shortcut Learning*.
3. **Thống kê độ dài văn bản**: Phân phối số từ (âm tiết) và số ký tự của `title` và `lead_paragraph`.
4. **Phân tích dấu câu & từ khóa đặc trưng**: Tỷ lệ xuất hiện dấu `?`, `!`, dấu trích dẫn `""`, chữ số, từ viết hoa toàn bộ (`ALL CAPS`) và trực quan hóa `WordCloud`.

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

sys.path.append(os.path.abspath('..'))
from src.evaluate import generate_eda_figures

df = pd.read_csv('../data/processed/full_processed.csv', encoding='utf-8')
print(f'Kích thước bộ dữ liệu: {df.shape}')
df[['id', 'source', 'category', 'title', 'lead_paragraph', 'label']].head()

## 1. Kiểm tra Missing Values và Phân phối Nhãn (`label`)

In [ ]:
raw_df = pd.read_csv('../data/raw/clickbait_dataset_vietnamese.csv', encoding='utf-8')
print('=== Kiểm tra Missing Values trên dữ liệu gốc ===')
print(raw_df.isnull().sum())
print('\n=== Phân phối nhãn ===')
print(df['label'].value_counts())
print((df['label'].value_counts(normalize=True) * 100).round(2))
display(Image(filename='../outputs/figures/01_label_distribution.png'))

## 2. Phân phối theo Nguồn báo (`source`) & Chuyên mục (`category`) - Cảnh báo Rò rỉ Dữ liệu (Data Leakage)
- **Nhận xét quan trọng**: Các trang tin giải trí/giới trẻ như **Kênh14** (76.92% clickbait) và **SaoStar** (65.03% clickbait) có tỷ lệ giật tít rất cao; ngược lại **VnExpress** (85.53% non-clickbait) và **Báo Mới** (79.75% non-clickbait) đa số là tin chính thống.
- **Quyết định thiết kế mô hình**: Tuyệt đối **không đưa cột `source` vào làm feature huấn luyện** và **cắt bỏ hậu tố tên báo trong tiêu đề Báo Mới** để ép mô hình học từ **ngữ nghĩa văn bản (`title` + `lead_paragraph`)** thay vì học vẹt tên báo (*Anti-Shortcut Learning*).

In [ ]:
source_ct = pd.crosstab(df['source'], df['label'], margins=True)
source_pct = (pd.crosstab(df['source'], df['label'], normalize='index') * 100).round(2)
print('=== Bảng chéo Nguồn báo vs Nhãn ===')
display(pd.concat([source_ct, source_pct.add_suffix(' (%)')], axis=1))
display(Image(filename='../outputs/figures/02_source_category_distribution.png'))

## 3. Thống kê Độ dài Văn bản (`title` & `lead_paragraph`)

In [ ]:
length_cols = ['title_word_count', 'title_char_len', 'lead_word_count', 'lead_char_len']
display(df.groupby('label')[length_cols].agg(['mean', 'std', 'median', 'min', 'max']).round(2))
display(Image(filename='../outputs/figures/03_text_length_distribution.png'))

## 4. Phân tích Dấu câu cảm thán, Từ khóa đặc trưng và WordCloud

In [ ]:
display(Image(filename='../outputs/figures/04_punctuation_features.png'))
display(Image(filename='../outputs/figures/05_wordcloud_comparison.png'))